In [2]:
import numpy as np
from numba import njit
import pandas as pd
import math
import heapq
from itertools import combinations

In [ ]:
a = pd.read_csv("./datasets/minitsp.csv")
points = np.array(a[['x','y']])
inputsize = len(points)

FileNotFoundError: [Errno 2] No such file or directory: './datasets/minitsp.csv'

In [ ]:
# distance matrix 만들기
diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
distance_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
len(distance_matrix)

280

In [ ]:
# distance_matrix = [
#     [0, 2, 9, 10],
#     [1, 0, 6, 4],
#     [15, 7, 0, 8],
#     [6, 3, 12, 0],
# ]


# def dp(S, v): #v 는 인덱스 S 는 리스트
#     Min = np.inf
#     size = len( S )
#     if size == 1:
#         return distance_matrix[0][v]
#     for i in S:
#         if i == v:
#             continue
#         new_s = S.copy()
#         new_s.remove(v)
#         Min = min (Min, dp( new_s, i ) + distance_matrix[i][v] )
#     return Min

# def dp(S, v=0): #v 는 인덱스 S 는 리스트
#     Min = np.inf
#     size = len( S )
#     minindex=0
#     if size == 1:
#         memo[0][v]=distance_matrix[0][v]
#         return distance_matrix[0][v]
#     Sbitmask = 0
#     for x in S:
#         Sbitmask += 1<<x
#     if np.isfinite( memo[Sbitmask][v] ):
#         return memo[Sbitmask][v]
#     for i in S:
#         if i == v:
#             continue
#         new_s = S.copy()
#         new_s.remove(v)
#         cost = dp( new_s, i ) + distance_matrix[i][v]
#         if cost < Min:
#             Min = cost
#             minindex = i
#         memo[Sbitmask - (1<<minindex)][v] = Min
#     return Min


S = distance_matrix
v = 0

memo = np.full((1 << len(S), len(S)), np.inf) 

def dp(S, v=0): #v 는 인덱스 S 는 리스트
    Min = np.inf
    size = len( S )
    minindex=0
    if size == 1:
        memo[0][v]=distance_matrix[0][v]
        return distance_matrix[0][v]
    Sbitmask = 0
    for x in S:
        Sbitmask += 1<<x
    if np.isfinite( memo[Sbitmask][v] ):
        return memo[Sbitmask][v]
    for i in S:
        if i == v:
            continue
        new_s = S.copy()
        new_s.remove(v)
        cost = dp( new_s, i ) + distance_matrix[i][v]
        if cost < Min:
            Min = cost
            minindex = i
    memo[Sbitmask][v] = Min
    return Min


x = [x for x in range(len(S))]
b = np.array(x)
dp(x,0)
# memo
# 246.8
# np.delete(b,2)
# memo

ValueError: Maximum allowed dimension exceeded

In [ ]:
import itertools

def held_karp_iter(dist_matrix):
    n = len(dist_matrix)
    if n == 0: return 0, []
    if n == 1: return 0, [0]

    C = {}

    for k in range(1, n):
        C[(1 << k, k)] = (dist_matrix[0][k], 0)

    for size in range(2, n):
        for subset_indices in itertools.combinations(range(1, n), size):
            S = 0
            for bit in subset_indices:
                S |= (1 << bit)
            for k in subset_indices:
                prev_S = S & ~(1 << k)
                min_val = float('inf')
                best_prev_node = -1
                for m in subset_indices:
                    if m == k: continue
                    if not (prev_S & (1 << m)):
                        if prev_S == 0 and m == 0:
                           pass
                        else:
                            continue

                    if (prev_S, m) in C:
                        cost, _ = C[(prev_S, m)]
                        if cost + dist_matrix[m][k] < min_val:
                            min_val = cost + dist_matrix[m][k]
                            best_prev_node = m
                if best_prev_node != -1:
                    C[(S, k)] = (min_val, best_prev_node)

    last_S_mask = 0
    for i in range(1,n):
        last_S_mask |= (1 << i)

    min_tour_len = float('inf')
    last_node_of_tour = -1

    if n == 1:
        return 0, [0]
    
    if not C and n > 1:
        if n == 2:
            min_tour_len = dist_matrix[0][1] + dist_matrix[1][0]
            last_node_of_tour = 1
        else:
            return float('inf'), []

    for k in range(1, n):
        if (last_S_mask, k) in C:
            cost, _ = C[(last_S_mask, k)]
            total_cost = cost + dist_matrix[k][0]
            if total_cost < min_tour_len:
                min_tour_len = total_cost
                last_node_of_tour = k
        elif n==2 and k==1:
             cost = dist_matrix[0][1]
             total_cost = cost + dist_matrix[1][0]
             if total_cost < min_tour_len:
                min_tour_len = total_cost
                last_node_of_tour = 1

    if last_node_of_tour == -1:
        if n > 0:
             return float('inf'), []
        else:
             return 0, []

    path = []
    curr_node = last_node_of_tour
    curr_mask = last_S_mask
    while curr_node != 0 :
        path.append(curr_node)
        if curr_mask == 0 or curr_node == -1 : break
        
        if curr_mask == (1 << curr_node):
            prev_node = 0
        elif (curr_mask, curr_node) in C:
             _, prev_node = C[(curr_mask, curr_node)]
        else:
            return float('inf'), []

        curr_mask &= ~(1 << curr_node)
        curr_node = prev_node
        if curr_node == 0 and curr_mask != 0:
            return float('inf'), []

    path.append(0)
    final_path = path[::-1]

    return min_tour_len, final_path


In [4]:
def dp_path(S, v, start, end, memo, dist):
    if len(S) == 0:
        return 0 if v == end else np.inf
    mask = sum(1 << x for x in S)
    
    if memo[mask][v] != np.inf:
        return memo[mask][v]
    
    min_cost = np.inf
    for next_node in S:
        if next_node == v:
            continue
            
        new_S = [x for x in S if x != next_node]
        cost = dp_path(new_S, next_node, start, end, memo, dist) + dist[v][next_node]
        
        if cost < min_cost:
            min_cost = cost
            
    memo[mask][v] = min_cost
    return min_cost


In [5]:
# 실행 예시
nodes = ['A','B','C','D']
node_idx = {n:i for i,n in enumerate(nodes)}
N = len(nodes)

# 거리 행렬 (예시)
dist_matrix = np.array([
    [0, 2, 3, 1],
    [2, 0, 4, 5],
    [3, 4, 0, 6],
    [1, 5, 6, 0]
])

# 메모 테이블 초기화
memo = np.full((1<<N, N), np.inf)

# 시작점(A)과 종료점(D) 설정
start_node = node_idx['A']
end_node = node_idx['D']
initial_S = [i for i in range(N) if i != start_node]

result = dp_path(initial_S, start_node, start_node, end_node, memo, dist_matrix)
print(f"최단 경로 거리: {result if result != np.inf else '경로 없음'}")

최단 경로 거리: 12


최종 경로 형태: (100, 2)
첫 5개 점: [[0.15601864 0.15599452]
 [0.18182497 0.18340451]
 [0.30461377 0.09767211]
 [0.0884925  0.19598286]
 [0.35846573 0.11586906]]


c:\Users\foxis\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 지정된 파일을 찾을 수 없습니다
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\foxis\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\foxis\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\foxis\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\foxis\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
 